# Joint masked encoder–decoder for bat hippocampus

A **single transformer with one shared latent space** that unifies the neural
*encoder* (`preprocess-data-v0j-share.ipynb`, behaviour → spikes) and the position
*decoder* (`decode-position-comparison.ipynb`, spikes → position). Following the
NEDS / POYO+ / OmniMouse lineage, encoding, decoding, infilling and forecasting are
all the *same* model under different **masking** schemes — no task-specific heads,
one set of weights.

**Design choices** (some overruling the auto-generated plan, per the brief):

* **Behaviour = 3-D position only** `(x, y, z)`.
* **Per-timestep, per-session tokenization** (NEDS-style), *not* POYO+ per-neuron
  tokens. The data are small and the model must be ~1 M params on 1 GPU; per-session
  linear read-in/read-out matches the existing `Bat_Transformer` / `Bat_Transformer_Decoder`
  and still yields interrogable per-neuron embeddings (read-in weight columns).
* **Two modality streams** (neural, behaviour), concatenated into a `2T` sequence;
  full bidirectional attention so each modality conditions on the other.
* **Timestamp-aware RoPE** (reuses the package `MultiheadAttentionRoPE`) so a neural
  and a behaviour token at the *same* time bin share rotary phase.
* **`ELU+1` (`ELU1`)** rate nonlinearity for the neural read-out, as in the existing
  encoding models; **Poisson NLL** for spikes, **MSE** for z-scored position.
* Transformer depth swept over **1, 2, 4, 6, 8, 10, 12 layers**.
* Generic metrics: encoding **corr / FDE** (not bits-per-spike); decoding
  **2-D / 3-D position error + discrete Fréchet** (not R²).

Masked tokens are replaced by a learned per-modality mask token and kept in the
sequence (MAE/NEDS style), so every task is a choice of which tokens are targets.

## 1 · Imports, device, seed

In [ ]:
import os, sys, math, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm

# digitalbat on path (notebooks/ -> repo root)
sys.path.insert(0, os.path.abspath('..'))

from digitalbat.datasets.data import bat_data_loader
from digitalbat.datasets.defaults import DEFAULT_SESSION_NAMES, make_data_config
from digitalbat.models.layers import MultiheadAttentionRoPE, ELU1
from digitalbat.decoding import discrete_frechet
from digitalbat.utility.utils import (
    corr, poisson_fde_population, round_robin_session_batches,
    parse_bat, bat_palette, set_random_state,
    wilcoxon_safe, stars_for_p, add_significance_bracket,
)

sns.set_context('notebook')
SEED = 0
set_random_state(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2 · Data

Standard `bat_data_loader` with the canonical config (`behavior_dim == 3`,
`bin_width = 0.10 s`, flight-only, raw spike counts). Batches are
`DataSample(behavior (B,T,3 z-scored), spike (B,T,K counts), mask (B,T),
flight_path_index, flight_type_index)`.

In [ ]:
# Datasets 1-3 only (no Krishna sessions): rooms are indoor lab vs Dataset 2.
data_config = make_data_config(session_names=DEFAULT_SESSION_NAMES)
dataloaders, behavior_dim, unit_num_dict, meta_info_dict = bat_data_loader(**data_config)

assert behavior_dim == 3, f'expected 3-D position, got behavior_dim={behavior_dim}'
sessions = list(unit_num_dict.keys())
print(f'{len(sessions)} sessions, behavior_dim={behavior_dim}, '
      f'{sum(unit_num_dict.values())} neurons total')
print('neurons/session:', {s: unit_num_dict[s] for s in sessions[:3]}, '...')

### 2.1 · Neuron metadata (for latent interrogation)

Build a per-neuron table (bat, session, probe, putative cell type, depth/lateral
position) so the learned neuron embeddings can be coloured by anatomy later.

In [ ]:
def build_neuron_meta(unit_num_dict, meta_info_dict):
    rows = []
    for s, k in unit_num_dict.items():
        ptype = np.asarray(meta_info_dict['pType'][s]).ravel()
        probe = np.asarray(meta_info_dict['probe_id'][s]).ravel()
        pos = np.asarray(meta_info_dict['neuron_position'][s])
        for j in range(k):
            rows.append(dict(
                session=s, bat=parse_bat(s), neuron_idx=j,
                pType=ptype[j] if j < len(ptype) else np.nan,
                probe_id=int(probe[j]) if j < len(probe) else -1,
                pos_lateral=float(pos[j, 0]) if pos.ndim == 2 and j < len(pos) else np.nan,
                pos_depth=float(pos[j, 1]) if pos.ndim == 2 and pos.shape[1] > 1 and j < len(pos) else np.nan,
            ))
    return pd.DataFrame(rows)

neuron_meta = build_neuron_meta(unit_num_dict, meta_info_dict)
print(neuron_meta.shape)
neuron_meta.head()

### 2.2 · Room map (different arenas have different geometry)

`(x, y, z)` is **not** a common frame across sessions: with Datasets 1-3, Dataset 2 is a
physically different arena from the Datasets 1/3 lab (bounds live in `meta_info_dict['r_lim']`),
so here the rooms are **`lab` vs `dataset_2`** (the `krishna_fs` branch only fires if Krishna
sessions are added). Per-session z-scoring (`behavior_mean`/`behavior_std`) already removes the
*scale* difference, but the same z-scored coordinate still means a different physical place per
arena, and place fields remap. Following the project convention (`neurons_across_envs.ipynb`,
`Bat_Transformer`'s `session_to_room`), we group sessions into **rooms** and make the
behaviour read-in/read-out **per-room** (§3) — the model's only across-room-shared maps;
neuron maps stay per-session and already carry room identity. `r_lim` is used for plot
limits only, never as model input. Set `session_to_room=None` for a single shared frame.

In [ ]:
def session_room(s):
    """3-config rooms (matches neurons_across_envs.ipynb session_room3):
    Krishna field station / Dataset 2 / indoor lab (Datasets 1 & 3 + Krishna lab)."""
    if '_fs' in s:
        return 'krishna_fs'
    if s.startswith('Dataset_2_'):
        return 'dataset_2'
    return 'lab'

session_to_room = {s: session_room(s) for s in sessions}
print({r: sum(v == r for v in session_to_room.values())
       for r in sorted(set(session_to_room.values()))})

## 3 · The joint model

One transformer over a `2T` token sequence: the first `T` tokens are **neural**
(per-session linear read-in of spike counts), the next `T` are **behaviour**
(**per-room** linear read-in of 3-D position; §2.2). Masked tokens become a learned
per-modality mask token. Per-session linear read-outs map the neural latent back to a
Poisson rate (`ELU+1`); a **per-room** read-out maps the behaviour latent back to position.

`build_rotary_emb_from_timestamps` gives both modalities matching rotary phase per
time bin (the packaged `build_rotary_emb` keys off sequence index, which would
mis-phase the concatenated sequence).

The schematic below summarises the data flow; the masking panel shows how a single
set of weights serves every task.

In [ ]:
# Architecture schematic (also saved to ./out/joint_architecture.png)
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def _draw_architecture(ax):
    C = dict(neural='#f4cccc', beh='#cfe2f3', shared='#d9ead3', core='#fff2cc',
             out='#ead1dc', note='#f3f3f3')
    def box(x, y, w, h, text, fc, fs=9):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.06',
                                    fc=fc, ec='#444', lw=1.2, zorder=2))
        ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fs, zorder=3)
    def arrow(p0, p1):
        ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=14,
                                     lw=1.6, color='#333', zorder=1))
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')
    box(1, 70, 15, 12, 'Spikes\n(B, T, K)\nraw counts', C['neural'])
    box(20, 70, 16, 12, 'Per-session\nLinear read-in\n(raw counts)', C['neural'])
    box(40, 70, 15, 12, '+ modality emb\n+ mask token\n(if target)', C['note'], 8.5)
    box(1, 18, 15, 12, 'Position\n(B, T, 3)\nz-scored xyz', C['beh'])
    box(20, 18, 16, 12, 'Per-room Linear\nread-in', C['beh'])
    box(40, 18, 15, 12, '+ modality emb\n+ mask token\n(if target)', C['note'], 8.5)
    box(59, 44, 17, 14, 'Concat -> (B, 2T, D)\n[ neural_1..T |\n  behaviour_1..T ]\ntimestamp RoPE', C['shared'], 8.5)
    box(80, 44, 18, 14, 'Joint transformer\nx 4 layers\nfull attention,\npre-norm + FFN', C['core'])
    box(80, 70, 18, 12, 'Per-session read-out\n-> ELU+1 -> Poisson\nrate (B, T, K)', C['out'], 8.5)
    box(80, 18, 18, 12, 'Per-room read-out\n-> position\n(B, T, 3)', C['out'], 8.5)
    for p0, p1 in [((16, 76), (20, 76)), ((36, 76), (40, 76)), ((16, 24), (20, 24)),
                   ((36, 24), (40, 24)), ((55, 74), (62, 57)), ((55, 26), (62, 45)),
                   ((76, 51), (80, 51)), ((89, 58), (89, 70)), ((89, 44), (89, 30))]:
        arrow(p0, p1)
    ax.text(89, 85.5, 'Poisson NLL  (masked neural targets)', ha='center', va='bottom',
            fontsize=8, color='#7a3b52')
    ax.text(89, 16.5, 'lam x MSE  (masked behaviour targets)', ha='center', va='top',
            fontsize=8, color='#3b5a7a')
    ax.text(2, 60, 'One model, task = choice of mask  (target = predicted):',
            fontsize=9, fontweight='bold')
    rows = [('encode', 'neural target | behaviour seen', 'spikes <- behaviour'),
            ('decode', 'neural seen | behaviour target', 'position <- spikes  (headline)'),
            ('within', 'random 50% each, independent', 'within-modality infill'),
            ('cross', 'random 50%, same tokens', 'cross-modal infill'),
            ('forecast', '2nd half of both = target', 'predict the future')]
    for i, (name, mask, task) in enumerate(rows):
        y = 55 - i * 3.0
        ax.text(3, y, name, fontsize=8.2, fontweight='bold', family='monospace', va='center')
        ax.text(15, y, mask, fontsize=8, va='center', color='#333')
        ax.text(40, y, '->  ' + task, fontsize=8, va='center', color='#555')

fig, ax = plt.subplots(figsize=(13.5, 7.4))
_draw_architecture(ax)
ax.set_title('Joint masked encoder-decoder (NEDS-style): per-timestep, per-session tokens; '
             'shared latent;\nencode / decode / infill / forecast via masking', fontsize=11)
os.makedirs('./out', exist_ok=True)
fig.savefig('./out/joint_architecture.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Architecture lives in the package (digitalbat/models/joint.py); imported here
# so the notebook and package stay in sync (definitions deduplicated).
from digitalbat.models.joint import (
    build_rotary_emb_from_timestamps, JointBlock, JointMaskedTransformer)

for nl in (1, 4, 12):
    _m = JointMaskedTransformer(unit_num_dict, num_layers=nl, session_to_room=session_to_room)
    print(f'num_layers={nl}: {sum(p.numel() for p in _m.parameters())/1e6:.3f} M params')
del _m

## 4 · Masking schemes

Six tasks (per-group behaviour masking collapses to all-or-nothing with a single
3-D-position group, so it is folded into `encode`). `*_target == 1` marks tokens
masked on input (to be predicted); padded timesteps are never targets.

| scheme | neural | behaviour | task |
|---|---|---|---|
| `encode`   | masked (target) | visible | spikes ← behaviour |
| `decode`   | visible | masked (target) | **position ← spikes** (headline) |
| `within`   | random 50 % | random 50 % (independent) | within-modality infilling |
| `cross`    | random 50 % | random 50 % (same tokens) | cross-modal infilling |
| `forecast` | 2nd half target | 2nd half target | predict the future |

In [ ]:
# Masking schemes + joint loss live in the package (digitalbat/models/joint.py).
from digitalbat.models.joint import SCHEMES, sample_mask, joint_loss
print('masking schemes:', SCHEMES)

## 5 · Trainer

Round-robin over sessions (`round_robin_session_batches`), one masking scheme
sampled per batch. Both validation metrics are tracked every epoch: the decode
median 3-D position error (metres, ↓) and the encode median FDE (↑).

**Model selection — joint criterion weighting FDE equally.** Rather than selecting
on decoding alone, we keep a **Pareto front** over `(−decode-error, +FDE)` during
training (a CPU checkpoint is stored only when an epoch is non-dominated, so memory
is tiny), then pick the front member maximising a **reference-scale** score

$$ J = w\,\frac{\mathrm{FDE}}{\mathrm{fde\_ref}} - (1-w)\,\frac{\mathrm{err}}{\mathrm{err\_ref}},\qquad w = 0.5. $$

Decode error is ~1 m but FDE is ~0.01, so dividing each by a typical magnitude
(`err_ref = 1.0 m`, `fde_ref = 0.01`) puts them on a common raw scale: a 0.01 gain
in FDE then counts as much as a 1 m gain in error — i.e. **FDE is up-weighted ~100×**
(the ratio `err_ref / fde_ref`). Tune `err_ref`/`fde_ref` to change the trade-off, or
`fde_weight` to tilt it further; `select='decode'` recovers decode-only selection.
(A std-based z-score would instead equalise by each metric's *spread*, giving a
data-dependent factor rather than a fixed, interpretable ~100×.)

`lam` weights the behaviour (MSE) loss vs the spike (Poisson) loss **during
training**. It is a fixed hyperparameter here (`lam=1.0`, *not* tuned); with z-scored
position the MSE term is ~O(1), comparable to the Poisson term. Selection is now
scale-free regardless of `lam`, so `lam` only sets the *gradient* balance. It is held at
`1.0` (not swept); §6.1 instead runs an Optuna search over `lr`, `weight_decay`,
`dropout`, `latent_dim` and `ff_dim` at fixed depth.

In [ ]:
@torch.no_grad()
def _decode_median_err_m(model, loaders, meta, device):
    """Mean over sessions of median 3-D decode error (metres) on a split."""
    model.eval()
    per_session = []
    for s, sd in loaders.items():
        std = np.asarray(meta['behavior_std'][s]).squeeze()[:3]
        errs = []
        for b in sd:
            m = b.mask.to(device)
            bt = b.mask.clone().to(device)            # decode: all behaviour masked
            nt = torch.zeros_like(bt)
            _, beh_pred, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            yp = beh_pred.cpu().numpy()
            yt = b.behavior[:, :, :3].numpy()
            mk = b.mask.numpy().astype(bool)
            resid = (yt - yp)[mk] * std[None, :]
            if resid.size:
                errs.append(np.sqrt((resid ** 2).sum(axis=1)))
        if errs:
            per_session.append(float(np.median(np.concatenate(errs))))
    return float(np.mean(per_session)) if per_session else np.inf


@torch.no_grad()
def _encode_median_fde(model, loaders, device):
    """Median FDE across neurons, encode task (spikes <- behaviour)."""
    model.eval()
    fdes = []
    for s, sd in loaders.items():
        yt_l, yp_l = [], []
        for b in sd:
            m = b.mask.to(device)
            nt = b.mask.clone().to(device)            # encode: all neural masked
            bt = torch.zeros_like(nt)
            rate, _, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            mk = b.mask.numpy().astype(bool)
            yt_l.append(b.spike.numpy()[mk]); yp_l.append(rate.cpu().numpy()[mk])
        if yt_l:
            fdes.append(poisson_fde_population(np.vstack(yt_l), np.vstack(yp_l)))
    return float(np.nanmedian(np.hstack(fdes))) if fdes else np.nan


def _cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def _pareto_update(cand, err, fde, model):
    """Maintain a Pareto front over (err down, fde up). Store a CPU checkpoint only
    when the new (err, fde) is non-dominated; drop members it dominates. Returns
    (cand, improved)."""
    dominated = any((o['err'] <= err and o['fde'] >= fde) and
                    (o['err'] < err or o['fde'] > fde) for o in cand)
    if dominated:
        return cand, False
    cand = [o for o in cand if not ((err <= o['err'] and fde >= o['fde']) and
                                     (err < o['err'] or fde > o['fde']))]
    cand.append(dict(err=err, fde=fde, w=_cpu_state(model)))
    return cand, True


def train_joint(model, dataloaders, device, n_epochs=300, lr=3e-3, weight_decay=1e-2,
                lam=1.0, schemes=SCHEMES, scheme_p=None, select='joint', fde_weight=0.5,
                err_ref=1.0, fde_ref=0.01, early_stopping_patience=40, verbose=True):
    """select='joint': pick the Pareto-front checkpoint maximising
    fde_weight*(FDE/fde_ref) - (1-fde_weight)*(err/err_ref). Dividing by reference
    magnitudes (err_ref m, fde_ref) puts the two on a common raw scale: with
    err_ref=1.0 and fde_ref=0.01 a 0.01 FDE gain counts as a 1 m error gain, i.e.
    FDE is effectively up-weighted ~100x (its raw scale is ~100x smaller). 'decode':
    legacy min-decode-error. Returns (model, best, hist) with best a dict of the
    selected epoch's err / fde / joint score."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    pois_fn = nn.PoissonNLLLoss(log_input=False, reduction='none').to(device)
    hist = {'val_err_m': [], 'val_fde': []}
    rng = random.Random(SEED)
    best_err, best_w, no_improve = np.inf, None, 0          # select='decode'
    cand = []                                               # select='joint' Pareto front
    for ep in tqdm.tqdm(range(n_epochs), desc='joint'):
        model.train()
        for s, b in round_robin_session_batches(dataloaders['train']):
            scheme = rng.choices(schemes, weights=scheme_p)[0]
            m = b.mask.to(device)
            nt, bt = sample_mask(b.mask, scheme)
            nt, bt = nt.to(device), bt.to(device)
            rate, beh, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            loss = joint_loss(rate, beh, b.spike.to(device), b.behavior.to(device),
                              m, nt, bt, lam, pois_fn)
            opt.zero_grad(); loss.backward(); opt.step()
        err = _decode_median_err_m(model, dataloaders['valid'], meta_info_dict, device)
        fde = _encode_median_fde(model, dataloaders['valid'], device)
        hist['val_err_m'].append(err); hist['val_fde'].append(fde)
        if select == 'decode':
            improved = err < best_err
            if improved:
                best_err, best_w = err, copy.deepcopy(model.state_dict())
        else:
            cand, improved = _pareto_update(cand, err, fde, model)
        no_improve = 0 if improved else no_improve + 1
        if early_stopping_patience and no_improve >= early_stopping_patience:
            if verbose: print(f'early stop @ epoch {ep+1}')
            break
    if select == 'joint':
        errs = np.array([o['err'] for o in cand], float)
        fdes = np.array([o['fde'] for o in cand], float)
        # reference-scale normalisation (common raw scale); fde_ref ~100x smaller
        # than err_ref => FDE up-weighted ~100x. higher = better.
        joint = fde_weight * (fdes / fde_ref) - (1.0 - fde_weight) * (errs / err_ref)
        bi = int(np.nanargmax(joint))
        model.load_state_dict(cand[bi]['w'])
        best = dict(err=float(errs[bi]), fde=float(fdes[bi]), joint=float(joint[bi]))
    else:
        if best_w is not None:
            model.load_state_dict(best_w)
        best = dict(err=float(best_err), fde=np.nan, joint=np.nan)
    return model, best, hist

## 6 · Training

Hyperparameters first (cell below), then an **Optuna search** (§6.1) over depth
(`num_layers` in `{1, 2, 4}`, started at 4), the optimiser and the capacity knobs at
fixed `LAM = 1.0`, then **train + cache** the single selected model (§6.2). Reduce
`N_EPOCHS` / `N_TRIALS` for a quick pass; bump them for the real run.

In [ ]:
N_EPOCHS = 1000
LAYER_CHOICES = [1, 2, 4]   # transformer depths searched by Optuna (§6.1)
START_LAYERS = 4            # first enqueued trial starts at depth 4
LATENT_DIM = 64            # default / Optuna starting point (search may revise latent/ff)
LAM = 1.0                  # behaviour-loss weight; FIXED (not swept)
INPUT_TRANSFORM = 'identity'   # raw spike counts into the read-in (no sqrt)
# selection reference scales (typical decode err / FDE) -> FDE up-weighted ~23x
ERR_REF, FDE_REF = 0.8, 0.035
N_TRIALS = 30             # Optuna trials (each trains one full joint model)
TRIAL_EPOCHS = 300       # epoch cap per Optuna trial (final model uses N_EPOCHS)

### 6.1 · Optuna hyperparameter search

`LAM = 1.0` is fixed (no behaviour-weight sweep); everything else is an **Optuna** TPE
search — mirroring `decode-position-comparison.ipynb` / `cebra-comparison.ipynb` — over
`num_layers` (depth, restricted to `{1, 2, 4}`), `lr`, `weight_decay`, `dropout`,
`latent_dim` and `ff_dim`. The search is **seeded with the current best config**
(depth 4, latent 64, `lr` 3e-3, `wd` 1e-2, dropout 0.2) as the first trial, then
explores around it. Each trial trains a joint model for up to `TRIAL_EPOCHS = 300` epochs (early
stopping may cut it shorter) and is scored on the validation **joint criterion**
(§5); the final selected model (§6.2) is then trained for the full `N_EPOCHS`. The study is cached to
`./out/joint/joint_optuna.pkl` and topped up to `N_TRIALS`.

In [ ]:
import optuna
import pickle

os.makedirs('./out/joint', exist_ok=True)
JOINT_STUDY_PATH = './out/joint/joint_optuna.pkl'


def build_joint_model(params):
    """Fresh JointMaskedTransformer at `params` (lam / input-transform fixed by §6)."""
    set_random_state(SEED)
    return JointMaskedTransformer(
        unit_num_dict, behavior_dim=3,
        latent_dim=params['latent_dim'], num_heads=4, ff_dim=params['ff_dim'],
        num_layers=params['num_layers'], dropout=params['dropout'],
        input_transform=INPUT_TRANSFORM, session_to_room=session_to_room)


def joint_objective(trial):
    params = dict(
        num_layers=trial.suggest_categorical('num_layers', LAYER_CHOICES),
        latent_dim=trial.suggest_categorical('latent_dim', [32, 64, 128]),
        ff_dim=trial.suggest_categorical('ff_dim', [32, 64, 128]),
        dropout=trial.suggest_float('dropout', 0.0, 0.4),
        lr=trial.suggest_float('lr', 1e-4, 3e-2, log=True),
        weight_decay=trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True),
    )
    model = build_joint_model(params)
    _, best, _ = train_joint(model, dataloaders, device, n_epochs=TRIAL_EPOCHS,
                             lr=params['lr'], weight_decay=params['weight_decay'],
                             lam=LAM, select='joint', fde_weight=0.5,
                             err_ref=ERR_REF, fde_ref=FDE_REF, verbose=False)
    trial.set_user_attr('err_m', best['err'])
    trial.set_user_attr('fde', best['fde'])
    return best['joint']                       # maximise validation joint score

# Seed the search with the current best-known config as the first trial.
START_PARAMS = dict(num_layers=START_LAYERS, latent_dim=64, ff_dim=64,
                    dropout=0.2, lr=3e-3, weight_decay=1e-2)

if os.path.exists(JOINT_STUDY_PATH):
    with open(JOINT_STUDY_PATH, 'rb') as f:
        joint_study = pickle.load(f)
    print(f'Loaded existing study with {len(joint_study.trials)} trials')
else:
    joint_study = optuna.create_study(
        direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    joint_study.enqueue_trial(START_PARAMS)

remaining = max(0, N_TRIALS - len(joint_study.trials))
if remaining > 0:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    joint_study.optimize(joint_objective, n_trials=remaining, show_progress_bar=True)
    with open(JOINT_STUDY_PATH, 'wb') as f:
        pickle.dump(joint_study, f)

best_params = dict(START_PARAMS)
best_params.update(joint_study.best_params)
print(f'\nBest validation joint score: {joint_study.best_value:.4f}')
print('Best params:', best_params)
joint_study.trials_dataframe().sort_values('value', ascending=False).head(10)

In [ ]:
# Optuna study diagnostics: optimization history + how the score depends on the knobs
trials_df = joint_study.trials_dataframe()
done = trials_df[trials_df['state'] == 'COMPLETE'].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(done['number'], done['value'], 'o', color='0.6', alpha=0.7)
axes[0].plot(done['number'], done['value'].cummax(), '-', color='tab:red', label='running best')
axes[0].set(xlabel='trial', ylabel='validation joint score', title='Optimization history')
axes[0].legend()

sc = axes[1].scatter(done['params_lr'], done['value'],
                     c=np.log10(done['params_weight_decay']), cmap='viridis')
axes[1].set(xscale='log', xlabel='lr', ylabel='joint score', title='score vs lr')
plt.colorbar(sc, ax=axes[1], label='log10 weight_decay')

sc2 = axes[2].scatter(done['user_attrs_err_m'], done['user_attrs_fde'],
                      c=done['value'], cmap='magma')
axes[2].set(xlabel='valid decode err (m)', ylabel='valid median FDE',
            title='trials in the err-FDE plane')
plt.colorbar(sc2, ax=axes[2], label='joint score')
fig.suptitle('Optuna search summary'); plt.tight_layout(); plt.show()

### 6.2 · Train the selected model

Train one joint model at the Optuna-selected hyperparameters (`best_params`, including
the selected depth `NUM_LAYERS`; `LAM = 1.0`, raw-count input), select it on the joint
criterion (§5), and **cache the weights** so re-runs skip training. The validation
curves below locate the plateau and mark the selected epoch.

In [ ]:
# Train (or load) the single selected joint model and cache its weights.
NUM_LAYERS = int(best_params['num_layers'])   # selected depth (downstream titles use this)
JOINT_CKPT = (f"./out/joint/joint_best_L{NUM_LAYERS}_d{best_params['latent_dim']}_"
              f"lam{LAM:g}_{INPUT_TRANSFORM}.pth")

joint_model = build_joint_model(best_params)
if os.path.exists(JOINT_CKPT):
    joint_model.load_state_dict(torch.load(JOINT_CKPT, map_location=device))
    joint_model = joint_model.to(device)
    joint_hist = None
    joint_best = dict(
        err=_decode_median_err_m(joint_model, dataloaders['valid'], meta_info_dict, device),
        fde=_encode_median_fde(joint_model, dataloaders['valid'], device))
    print(f'loaded cached joint model from {JOINT_CKPT}')
else:
    joint_model, joint_best, joint_hist = train_joint(
        joint_model, dataloaders, device, n_epochs=N_EPOCHS,
        lr=best_params['lr'], weight_decay=best_params['weight_decay'],
        lam=LAM, select='joint', fde_weight=0.5, err_ref=ERR_REF, fde_ref=FDE_REF)
    torch.save(joint_model.state_dict(), JOINT_CKPT)
    print(f'saved joint model to {JOINT_CKPT}')

print(f'selected depth={NUM_LAYERS}; validation decode err={joint_best["err"]:.3f} m  '
      f'FDE={joint_best["fde"]:.3f}')

In [ ]:
# Validation curves vs epoch for the selected model (skipped when loaded from cache).
if joint_hist is not None:
    jt = (0.5 * np.array(joint_hist['val_fde'], float) / FDE_REF
          - 0.5 * np.array(joint_hist['val_err_m'], float) / ERR_REF)
    sel = int(np.nanargmax(jt))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for ax, (y, lab) in zip(axes, [(joint_hist['val_err_m'], 'valid 3-D decode error (m)  (down)'),
                                   (joint_hist['val_fde'], 'valid median FDE  (up)'),
                                   (jt, 'joint selection score  (up)')]):
        ax.plot(y, color='tab:purple')
        ax.plot(sel, np.asarray(y)[sel], 'o', color='gold', ms=9, mec='k', mew=0.6, zorder=5)
        ax.set(xlabel='epoch', ylabel=lab)
    fig.suptitle(f'Validation curves (num_layers={NUM_LAYERS}); selected epoch marked')
    plt.tight_layout(); plt.show()
else:
    print('cached model loaded — no training history to plot')

## 7 · Encoding evaluation — corr & FDE

Encode task on the held-out test split: spikes ← behaviour. Per-neuron correlation
(`corr`) and Poisson fraction-deviance-explained (`poisson_fde_population`), pooled
over valid test timesteps per session. FDE raw values are kept for stats; the plot
axis is clipped to `[-1, 1]` (FDE is unbounded below).

In [ ]:
@torch.no_grad()
def encoding_records(model, loaders, device):
    model.eval()
    rows = []
    for s, sd in loaders.items():
        yt_l, yp_l = [], []
        for b in sd:
            m = b.mask.to(device)
            nt = b.mask.clone().to(device); bt = torch.zeros_like(nt)
            rate, _, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            mk = b.mask.numpy().astype(bool)
            yt_l.append(b.spike.numpy()[mk]); yp_l.append(rate.cpu().numpy()[mk])
        if not yt_l:
            continue
        yt, yp = np.vstack(yt_l), np.vstack(yp_l)
        c = corr(yt, yp, axis=0)
        f = poisson_fde_population(yt, yp)
        for j in range(yt.shape[1]):
            rows.append(dict(session=s, bat=parse_bat(s), neuron_idx=j,
                             corr=float(c[j]), fde=float(f[j])))
    return pd.DataFrame(rows)


# Single selected joint model (§6.2), encode task on the held-out test split.
enc_df = encoding_records(joint_model, dataloaders['test'], device)
print(f"encoding (num_layers={NUM_LAYERS}): "
      f"corr mean={enc_df['corr'].mean():.3f}  FDE median={enc_df['fde'].median():.3f}")
enc_df.head()

In [ ]:
# Per-session encoding metrics, coloured by bat. FDE uses the default y-range
# (per-session medians are well-behaved; no [-1, 1] axis clip).
bpal = bat_palette()
enc_sess = (enc_df.groupby(['bat', 'session'])
            .agg(corr=('corr', 'mean'), fde=('fde', 'median')).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (col, label) in zip(axes, [('corr', 'encoding corr (per-session mean)'),
                                   ('fde', 'encoding FDE (per-session median)')]):
    sns.boxplot(data=enc_sess, y=col, color='0.85', width=0.4, fliersize=0, ax=ax)
    sns.stripplot(data=enc_sess, y=col, hue='bat', palette=bpal, size=8, jitter=0.15,
                  edgecolor='k', linewidth=0.4, ax=ax, legend=(ax is axes[1]))
    ax.set(xlabel='', ylabel=label)
axes[1].legend(title='bat', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
fig.suptitle(f'Encoding (num_layers={NUM_LAYERS})'); plt.tight_layout(); plt.show()

## 8 · Decoding evaluation — position error & Fréchet (headline)

Decode task on held-out test: position ← spikes. Predictions are de-z-scored to
metres with `behavior_mean`/`behavior_std`. Per-timestep 2-D/3-D Euclidean error and
per-flight discrete Fréchet distance, grouped by `flight_path_index`.

In [ ]:
@torch.no_grad()
def decoding_records(model, loaders, meta, device):
    model.eval()
    rows = []
    for s, sd in loaders.items():
        bm = np.asarray(meta['behavior_mean'][s]).squeeze()[:3]
        bs = np.asarray(meta['behavior_std'][s]).squeeze()[:3]
        for b in sd:
            m = b.mask.to(device)
            bt = b.mask.clone().to(device); nt = torch.zeros_like(bt)
            _, beh_pred, _ = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            yp_z = beh_pred.cpu().numpy(); yt_z = b.behavior[:, :, :3].numpy()
            mk = b.mask.numpy().astype(bool)
            yt = yt_z * bs[None, None, :] + bm[None, None, :]
            yp = yp_z * bs[None, None, :] + bm[None, None, :]
            B, T = mk.shape
            fpi = np.broadcast_to(b.flight_path_index.numpy()[:, None], (B, T))
            tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
            ytf, ypf = yt[mk], yp[mk]
            err2 = np.sqrt(((ytf[:, :2] - ypf[:, :2]) ** 2).sum(1))
            err3 = np.sqrt(((ytf - ypf) ** 2).sum(1))
            fpf, tif = fpi[mk], tix[mk]
            for i in range(len(err2)):
                rows.append(dict(session=s, bat=parse_bat(s),
                                 flight_id=int(fpf[i]), t_idx=int(tif[i]),
                                 x_true=ytf[i, 0], y_true=ytf[i, 1], z_true=ytf[i, 2],
                                 x_pred=ypf[i, 0], y_pred=ypf[i, 1], z_pred=ypf[i, 2],
                                 err_2d=float(err2[i]), err_3d=float(err3[i])))
    return pd.DataFrame(rows)


def frechet_per_flight(df):
    rows = []
    for (bat, s, fid), g in df.groupby(['bat', 'session', 'flight_id'], sort=False):
        g = g.sort_values('t_idx')
        truth = g[['x_true', 'y_true', 'z_true']].to_numpy(float)
        pred = g[['x_pred', 'y_pred', 'z_pred']].to_numpy(float)
        rows.append(dict(bat=bat, session=s, flight_id=int(fid), n_t=len(g),
                         frechet_2d=discrete_frechet(truth[:, :2], pred[:, :2]),
                         frechet_3d=discrete_frechet(truth, pred)))
    return pd.DataFrame(rows)


# Single selected joint model, decode task on the held-out test split.
dec_ts_df = decoding_records(joint_model, dataloaders['test'], meta_info_dict, device)
dec_fl_df = frechet_per_flight(dec_ts_df)
print('decode median (m):', dec_ts_df[['err_2d', 'err_3d']].median().round(3).to_dict())
print('frechet median (m):', dec_fl_df[['frechet_2d', 'frechet_3d']].median().round(3).to_dict())

In [ ]:
# Per-session decoding medians, coloured by bat (single selected joint model).
metrics = [('err_2d', dec_ts_df, '2-D error (m)'), ('err_3d', dec_ts_df, '3-D error (m)'),
           ('frechet_2d', dec_fl_df, '2-D Fréchet (m)'), ('frechet_3d', dec_fl_df, '3-D Fréchet (m)')]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (col, src, label) in zip(axes, metrics):
    med = src.groupby(['bat', 'session'])[col].median().reset_index()
    sns.boxplot(data=med, y=col, color='0.85', width=0.4, fliersize=0, ax=ax)
    sns.stripplot(data=med, y=col, hue='bat', palette=bpal, size=8, jitter=0.15,
                  edgecolor='k', linewidth=0.4, ax=ax, legend=(ax is axes[-1]))
    ax.set(xlabel='', ylabel=label)
axes[-1].legend(title='bat', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
fig.suptitle(f'Decoding (num_layers={NUM_LAYERS})'); plt.tight_layout(); plt.show()

In [ ]:
# Example decoded flights (single selected joint model).
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, s in zip(axes, list(dec_ts_df['session'].unique())[:3]):
    g = dec_ts_df[dec_ts_df['session'] == s]
    for fid, gf in list(g.groupby('flight_id'))[:6]:
        gf = gf.sort_values('t_idx')
        ax.plot(gf['x_true'], gf['y_true'], color='k', lw=1.2, alpha=0.7)
        ax.plot(gf['x_pred'], gf['y_pred'], color='tab:red', lw=1.0, alpha=0.7)
    ax.set(title=f'{parse_bat(s)} · {s.split("_")[-1]}', xlabel='x (m)', ylabel='y (m)')
    ax.set_aspect('equal', 'box')
axes[0].plot([], [], 'k', label='true'); axes[0].plot([], [], 'tab:red', label='decoded')
axes[0].legend()
fig.suptitle(f'Decoded flight paths (num_layers={NUM_LAYERS})'); plt.tight_layout(); plt.show()

## 9 · Interrogating the shared latent

We read the **pre-read-out latent** — the transformer output one linear map before
either head — on the **held-out test** flights, under the two headline tasks:

1. **Encode** (behaviour → neural): the *neural* latent `h_n = x[:, :T]` that feeds
   `spike_readout`, with the neural tokens masked and behaviour visible. This is the
   spike-predictive geometry; we project it to 3-D and colour by behaviour, mirroring
   the latent viewer in `preprocess-data-v0j-share.ipynb`.
2. **Decode** (neural → behaviour): the *behaviour* latent `h_b = x[:, T:]` that feeds
   `beh_readout`, with the behaviour tokens masked and spikes visible — the
   position-predictive geometry, mirroring `decode-position-comparison.ipynb`.

For each we fit PCA-3 / UMAP-3 across all test timesteps and write an interactive
`Scatter3d` HTML (per-flight polylines, dropdown to switch the colour variable) to
`./out/joint/`. A final cell measures **cross-modal attention** — how much mass a
neural query token places on behaviour keys.

In [ ]:
from sklearn.decomposition import PCA
try:
    from umap import UMAP
    _have_umap = True
except Exception:
    _have_umap = False


@torch.no_grad()
def collect_pre_readout_latent(model, loaders, meta, task, device):
    """Pre-read-out latent on a split. task='encode' -> neural latent h_n (spikes
    masked, behaviour seen); task='decode' -> behaviour latent h_b (behaviour
    masked, spikes seen). Returns (L (N, D), meta_df) with de-z-scored position,
    within-flight time index and speed for colouring."""
    model.eval(); Ls, rows = [], []
    for s, sd in loaders.items():
        bm = np.asarray(meta['behavior_mean'][s]).squeeze()[:3]
        bs = np.asarray(meta['behavior_std'][s]).squeeze()[:3]
        for b in sd:
            m = b.mask.to(device); T = b.spike.shape[1]
            if task == 'encode':
                nt = b.mask.clone().to(device); bt = torch.zeros_like(nt)
            else:
                bt = b.mask.clone().to(device); nt = torch.zeros_like(bt)
            _, _, xfull = model(s, b.spike.to(device), b.behavior.to(device), m, nt, bt)
            h = (xfull[:, :T] if task == 'encode' else xfull[:, T:]).cpu().numpy()
            mk = b.mask.numpy().astype(bool)
            pos = b.behavior[:, :, :3].numpy() * bs[None, None, :] + bm[None, None, :]
            tix = np.broadcast_to(np.arange(T)[None, :], mk.shape)
            fpi = b.flight_path_index.numpy()
            for bi in range(mk.shape[0]):
                vm = mk[bi]
                if vm.sum() < 2:
                    continue
                Hi, Pi, Ti = h[bi][vm], pos[bi][vm], tix[bi][vm]
                spd = np.r_[0.0, np.linalg.norm(np.diff(Pi, axis=0), axis=1)]
                Ls.append(Hi)
                for k in range(len(Hi)):
                    rows.append(dict(session=s, bat=parse_bat(s), flight_key=int(fpi[bi]),
                                     t_idx=int(Ti[k]), x=Pi[k, 0], y=Pi[k, 1], z=Pi[k, 2],
                                     rel_t=int(Ti[k]), speed=float(spd[k])))
    return np.vstack(Ls), pd.DataFrame(rows)


def embed_3d(L, max_pts=30000):
    """PCA-3 (+ UMAP-3 if available) on row-subsampled L; returns dict + the kept
    row index (so colouring metadata can be sliced to match)."""
    idx = (np.sort(np.random.default_rng(SEED).choice(len(L), max_pts, replace=False))
           if len(L) > max_pts else np.arange(len(L)))
    out = {'PCA': PCA(n_components=3, random_state=SEED).fit_transform(L[idx])}
    if _have_umap:
        out['UMAP'] = UMAP(n_components=3, n_neighbors=30, min_dist=0.3,
                           random_state=SEED).fit_transform(L[idx])
    return out, idx


L_enc, md_enc = collect_pre_readout_latent(joint_model, dataloaders['test'], meta_info_dict, 'encode', device)
L_dec, md_dec = collect_pre_readout_latent(joint_model, dataloaders['test'], meta_info_dict, 'decode', device)
emb_enc, idx_enc = embed_3d(L_enc)
emb_dec, idx_dec = embed_3d(L_dec)
md_enc_s, md_dec_s = md_enc.iloc[idx_enc].reset_index(drop=True), md_dec.iloc[idx_dec].reset_index(drop=True)
print(f'encode latent: {L_enc.shape}  plotted {len(idx_enc)};  '
      f'decode latent: {L_dec.shape}  plotted {len(idx_dec)}')

# quick matplotlib look (PCA, coloured by position x) for both tasks
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (emb, md, ttl) in zip(axes, [(emb_enc, md_enc_s, 'encode latent (h_n)'),
                                      (emb_dec, md_dec_s, 'decode latent (h_b)')]):
    sc = ax.scatter(emb['PCA'][:, 0], emb['PCA'][:, 1], c=md['x'], cmap='viridis', s=4)
    ax.set(xlabel='PC1', ylabel='PC2', title=ttl); plt.colorbar(sc, ax=ax, label='x (m)')
fig.suptitle('Pre-read-out latent geometry (test data, PCA)'); plt.tight_layout(); plt.show()

In [ ]:
# Interactive 3-D HTML of the pre-read-out latent: per-flight polylines, one panel
# per dim-reduction, with a dropdown to switch the colour variable. Same idea as the
# latent viewers in preprocess-data-v0j-share.ipynb (encode) and
# decode-position-comparison.ipynb (decode).
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLOR_VARS = ['x', 'y', 'z', 'rel_t', 'speed']


def write_latent_3d_html(emb_methods, md, path, title, color_vars=COLOR_VARS):
    """emb_methods: {name: (N,3) array aligned to md rows}. Draws None-separated
    per-flight polylines per method; dropdown restyles line.color across panels."""
    flights = list(md.groupby(['session', 'flight_key'], sort=False).indices.values())

    def sep(v):                                   # NaN-separate flights into one array
        out = []
        for ix in flights:
            out.extend(v[ix].tolist()); out.append(np.nan)
        return np.asarray(out, float)

    col_sep = {cv: sep(md[cv].to_numpy(float)) for cv in color_vars}
    names = list(emb_methods)
    fig = make_subplots(rows=1, cols=len(names),
                        specs=[[{'type': 'scatter3d'}] * len(names)],
                        subplot_titles=names)
    c0 = color_vars[0]
    for ci, nm in enumerate(names):
        emb = emb_methods[nm]
        fig.add_trace(go.Scatter3d(
            x=sep(emb[:, 0]), y=sep(emb[:, 1]), z=sep(emb[:, 2]), mode='lines',
            line=dict(color=col_sep[c0], colorscale='Viridis', width=3,
                      showscale=(ci == len(names) - 1),
                      colorbar=(dict(title=c0, x=1.0, len=0.8, thickness=12)
                                if ci == len(names) - 1 else None)),
            name=nm, showlegend=False), row=1, col=ci + 1)
    buttons = [dict(method='restyle', label=cv,
                    args=[{'line.color': [col_sep[cv]] * len(names)}]) for cv in color_vars]
    fig.update_layout(title=title, height=600, width=520 * len(names),
                      updatemenus=[dict(buttons=buttons, x=0.0, y=1.12, xanchor='left',
                                        showactive=True)])
    fig.write_html(path)
    print('wrote', path)
    return fig


enc_html = write_latent_3d_html(
    emb_enc, md_enc_s, './out/joint/latent_encode_3d.html',
    'Encode latent h_n (behaviour → neural), test data')
dec_html = write_latent_3d_html(
    emb_dec, md_dec_s, './out/joint/latent_decode_3d.html',
    'Decode latent h_b (neural → behaviour), test data')
enc_html.show()
dec_html.show()

In [ ]:
# Cross-modal attention mass: among a neural query token's attention, what fraction
# lands on behaviour keys vs neural keys? The packaged attention's weight-returning
# path multiplies 0 x (NaN padded value), which corrupts deep stacks with padding,
# so we trim each batch to its no-padding window first (padding is always at the end)
# and run with everything visible (pure fusion).
@torch.no_grad()
def cross_modal_fraction(model, loaders, device, max_batches=8):
    model.eval()
    fracs, seen = [], 0
    for s, sd in loaders.items():
        for b in sd:
            L = int(b.mask.sum(1).min().item())                  # common valid prefix
            if L < 2:
                continue
            spk = b.spike[:, :L].to(device); beh = b.behavior[:, :L].to(device)
            vm = torch.ones(spk.shape[0], L, device=device)
            zt = torch.zeros_like(vm)
            _, _, _, attn = model(s, spk, beh, vm, zt, zt, return_attn_weights=True)
            A = attn[-1].mean(1)                                  # (B, 2L, 2L), head-avg, last layer
            cross = A[:, :L, L:].sum(-1)                          # neural query -> behaviour keys
            within = A[:, :L, :L].sum(-1)                         # neural query -> neural keys
            frac = cross / (cross + within).clamp(min=1e-8)
            fracs.append(float(np.nanmean(frac.cpu().numpy())))
            seen += 1
            if seen >= max_batches:
                break
        if seen >= max_batches:
            break
    return float(np.nanmean(fracs)) if fracs else np.nan


cmf = cross_modal_fraction(joint_model, dataloaders['test'], device)
print(f'cross-modal attention fraction (neural query -> behaviour keys, '
      f'num_layers={NUM_LAYERS}): {cmf:.3f}')

## 10 · Comparison to the standalone single-task transformers

Does *one* masked multi-task model match the **dedicated** single-task transformers?
Two references per task:

* **matched-capacity baselines** trained here — `Bat_Transformer` (encoder) and
  `Bat_Transformer_Decoder` (decoder) at the joint model's capacity
  (`latent_dim=LATENT_DIM`, `num_layers=NUM_LAYERS`, raw-count input). Trained on the
  same splits with their canonical objectives, then **cached** to `./out/joint/`.
* **optimized models** loaded directly from the other notebooks' checkpoints — the
  encoder from `preprocess-data-v0j-share.ipynb` (`_acc50_full_v2`) and the
  Optuna-tuned decoder from `decode-position-comparison.ipynb` — at their **native
  architectures** (no capacity matching). They were fit on the 28-session superset,
  but the per-session train/valid/test split is internally seeded
  (`np.random.seed(166)` in `split_flight_cluster`), so these 18 sessions' test
  flights are identical to ours — loading with `strict=False` keeps the shared trunk +
  the 18 matching session heads and there is **no train/test leakage**.

All models are evaluated with the same metric functions on the same test flights and
overlaid (joint vs matched vs optimized).

In [ ]:
from digitalbat.models.former import Bat_Transformer
from digitalbat.models.former_defaults import make_model_config
from digitalbat.models.decoder import Bat_Transformer_Decoder, bat_decoder_trainer
from digitalbat.training.trainer import bat_former_trainer

# Match the joint model's *selected* capacity (depth + width from §6.1).
MATCH_LATENT = int(best_params['latent_dim'])
MATCH_FF = int(best_params['ff_dim'])
ENC_BASE_CKPT = f'./out/joint/baseline_encoder_L{NUM_LAYERS}_d{MATCH_LATENT}.pth'
DEC_BASE_CKPT = f'./out/joint/baseline_decoder_L{NUM_LAYERS}_d{MATCH_LATENT}_{INPUT_TRANSFORM}.pth'


def _neg_euclid_z(y_true, y_pred):
    return -np.sqrt(((y_true - y_pred) ** 2).sum(axis=1))


# ---- matched-capacity ENCODER: Bat_Transformer (behaviour -> spikes) ----
enc_cfg = make_model_config(behavior_dim, unit_num_dict, meta_info_dict,
                            session_to_room=session_to_room)
enc_cfg.update(latent_dim=MATCH_LATENT, ff_dim=MATCH_FF, num_heads=4,
               num_layers=NUM_LAYERS, dropout_prob=0.2)
set_random_state(SEED)
encoder_only = Bat_Transformer(**enc_cfg)
if os.path.exists(ENC_BASE_CKPT):
    encoder_only.load_state_dict(torch.load(ENC_BASE_CKPT, map_location=device))
    encoder_only = encoder_only.to(device)
    print(f'loaded cached matched encoder {ENC_BASE_CKPT}')
else:
    bat_former_trainer(encoder_only, dataloaders, device, n_epochs=N_EPOCHS,
                       lr=1e-2, weight_decay=1e-2,
                       loss_fn=nn.PoissonNLLLoss(log_input=False),
                       valid_fn=poisson_fde_population, valid_stat='median',
                       early_stopping_patience=40, save_model=False)
    torch.save(encoder_only.state_dict(), ENC_BASE_CKPT)
    print(f'trained + cached matched encoder {ENC_BASE_CKPT}')

# ---- matched-capacity DECODER: Bat_Transformer_Decoder (spikes -> position) ----
set_random_state(SEED)
decoder_only = Bat_Transformer_Decoder(input_dim_dict=unit_num_dict, output_dim=3,
                                       latent_dim=MATCH_LATENT, num_heads=4, ff_dim=MATCH_FF,
                                       num_layers=NUM_LAYERS, dropout_prob=0.2,
                                       input_transform=INPUT_TRANSFORM)
if os.path.exists(DEC_BASE_CKPT):
    decoder_only.load_state_dict(torch.load(DEC_BASE_CKPT, map_location=device))
    decoder_only = decoder_only.to(device)
    print(f'loaded cached matched decoder {DEC_BASE_CKPT}')
else:
    bat_decoder_trainer(decoder_only, dataloaders, device, pos_dims=3, n_epochs=N_EPOCHS,
                        lr=3e-3, weight_decay=1e-2, loss_fn=nn.MSELoss(),
                        valid_fn=_neg_euclid_z, valid_stat='median',
                        early_stopping_patience=40, save_model=False)
    torch.save(decoder_only.state_dict(), DEC_BASE_CKPT)
    print(f'trained + cached matched decoder {DEC_BASE_CKPT}')

print(f'matched-capacity baselines ready (depth={NUM_LAYERS}, latent={MATCH_LATENT}, ff={MATCH_FF})')

In [ ]:
import pickle

# Metric functions for the standalone models (their forwards differ from the joint one).
@torch.no_grad()
def encoding_records_former(former, loaders, device):
    former.eval(); rows = []
    for s, sd in loaders.items():
        yt_l, yp_l = [], []
        for b in sd:
            yp = former(s, b.behavior.to(device), b.mask.to(device))
            mk = b.mask.numpy().astype(bool)
            yt_l.append(b.spike.numpy()[mk]); yp_l.append(yp.cpu().numpy()[mk])
        if not yt_l:
            continue
        yt, yp = np.vstack(yt_l), np.vstack(yp_l)
        c = corr(yt, yp, axis=0); f = poisson_fde_population(yt, yp)
        for j in range(yt.shape[1]):
            rows.append(dict(session=s, bat=parse_bat(s), neuron_idx=j,
                             corr=float(c[j]), fde=float(f[j])))
    return pd.DataFrame(rows)


@torch.no_grad()
def decoding_records_decoder(dec, loaders, meta, device):
    dec.eval(); rows = []
    for s, sd in loaders.items():
        bm = np.asarray(meta['behavior_mean'][s]).squeeze()[:3]
        bs = np.asarray(meta['behavior_std'][s]).squeeze()[:3]
        for b in sd:
            yp_z = dec(s, b.spike.to(device), b.mask.to(device)).cpu().numpy()
            yt_z = b.behavior[:, :, :3].numpy(); mk = b.mask.numpy().astype(bool)
            yt = yt_z * bs[None, None, :] + bm[None, None, :]
            yp = yp_z * bs[None, None, :] + bm[None, None, :]
            B, T = mk.shape
            fpi = np.broadcast_to(b.flight_path_index.numpy()[:, None], (B, T))
            tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
            ytf, ypf, fpf, tif = yt[mk], yp[mk], fpi[mk], tix[mk]
            e2 = np.sqrt(((ytf[:, :2] - ypf[:, :2]) ** 2).sum(1))
            e3 = np.sqrt(((ytf - ypf) ** 2).sum(1))
            for i in range(len(e2)):
                rows.append(dict(session=s, bat=parse_bat(s), flight_id=int(fpf[i]), t_idx=int(tif[i]),
                                 x_true=ytf[i, 0], y_true=ytf[i, 1], z_true=ytf[i, 2],
                                 x_pred=ypf[i, 0], y_pred=ypf[i, 1], z_pred=ypf[i, 2],
                                 err_2d=float(e2[i]), err_3d=float(e3[i])))
    return pd.DataFrame(rows)


# ---- optimized ENCODER (preprocess-data-v0j-share.ipynb / model_comparison.ipynb) ----
# Native default architecture (latent 32, 1 layer, ELU+1); load with strict=False so
# only the 18 shared sessions' readout heads (+ shared trunk) transfer.
ENC_OPT_CKPT = './out/bat_former/bat_former_seed_1_lr_1.00e-02_wd_1.00e-02_acc50_full_v2.pth'
encoder_opt = Bat_Transformer(**make_model_config(behavior_dim, unit_num_dict, meta_info_dict))
_emiss, _eunexp = encoder_opt.load_state_dict(
    torch.load(ENC_OPT_CKPT, map_location=device), strict=False)
_emiss_sess = sorted({k.split('.')[1] for k in _emiss if k.startswith('readouts.')})
assert not _emiss_sess, f'optimized encoder missing readouts for sessions: {_emiss_sess}'
encoder_opt = encoder_opt.to(device)
_dropped = len({k.split('.')[1] for k in _eunexp if k.startswith('readouts.')})
print(f'loaded optimized encoder ({os.path.basename(ENC_OPT_CKPT)}); '
      f'ignored {_dropped} non-shared readout heads')

# ---- optimized DECODER (decode-position-comparison.ipynb, Optuna metersearch) ----
with open('./out/bat_decoder/bat_decoder_seed_1_optuna_metersearch.pkl', 'rb') as f:
    _dec_study = pickle.load(f)
_op = _dec_study.best_params


def _params_to_substr(params):
    return '_'.join(f'{k}_{v:.4g}' if isinstance(v, float) else f'{k}_{v}'
                    for k, v in params.items())


DEC_OPT_CKPT = f'./out/bat_decoder/bat_decoder_seed_1_{_params_to_substr(_op)}.pth'
decoder_opt = Bat_Transformer_Decoder(
    input_dim_dict=unit_num_dict, output_dim=3, num_heads=4, final_activation=nn.Identity(),
    latent_dim=_op['latent_dim'], ff_dim=_op['ff_dim'], num_layers=_op['num_layers'],
    dropout_prob=_op['dropout_prob'], readin_dropout=_op['readin_dropout'],
    input_transform=_op['input_transform'])
_dmiss, _dunexp = decoder_opt.load_state_dict(
    torch.load(DEC_OPT_CKPT, map_location=device), strict=False)
_dmiss_sess = sorted({k.split('.')[1] for k in _dmiss if k.startswith('readins.')})
assert not _dmiss_sess, f'optimized decoder missing readins for sessions: {_dmiss_sess}'
decoder_opt = decoder_opt.to(device)
print(f'loaded optimized decoder ({os.path.basename(DEC_OPT_CKPT)}); native arch: '
      f"latent={_op['latent_dim']}, ff={_op['ff_dim']}, layers={_op['num_layers']}, "
      f"transform={_op['input_transform']}")

# ---- evaluate everything on the joint test split and assemble comparison frames ----
enc_cmp = pd.concat([
    enc_df.assign(model='joint'),
    encoding_records_former(encoder_only, dataloaders['test'], device).assign(model='encoder-only'),
    encoding_records_former(encoder_opt, dataloaders['test'], device).assign(model='encoder-opt'),
], ignore_index=True)

dec_only_ts = decoding_records_decoder(decoder_only, dataloaders['test'], meta_info_dict, device)
dec_opt_ts = decoding_records_decoder(decoder_opt, dataloaders['test'], meta_info_dict, device)
dec_cmp_ts = pd.concat([dec_ts_df.assign(model='joint'),
                        dec_only_ts.assign(model='decoder-only'),
                        dec_opt_ts.assign(model='decoder-opt')], ignore_index=True)
fre_cmp = pd.concat([dec_fl_df.assign(model='joint'),
                     frechet_per_flight(dec_only_ts).assign(model='decoder-only'),
                     frechet_per_flight(dec_opt_ts).assign(model='decoder-opt')], ignore_index=True)

print('encoding (corr mean / FDE median):')
print(enc_cmp.groupby('model').agg(corr=('corr', 'mean'), fde=('fde', 'median')).round(3))
print('\ndecoding (median m):')
print(dec_cmp_ts.groupby('model')[['err_2d', 'err_3d']].median().round(3))

In [ ]:
ENC_ORDER = ['encoder-opt', 'encoder-only', 'joint']
DEC_ORDER = ['decoder-opt', 'decoder-only', 'joint']


def _compare_box(ax, med, col, label, order, ref='joint', clip=None):
    order = [m for m in order if m in set(med['model'])]
    sns.boxplot(data=med, x='model', y=col, order=order, color='0.85', width=0.5,
                fliersize=0, ax=ax)
    sns.stripplot(data=med, x='model', y=col, order=order, hue='bat', palette=bpal,
                  size=7, jitter=0.15, edgecolor='k', linewidth=0.4, ax=ax, legend=False)
    piv = med.pivot_table(index='session', columns='model', values=col)
    if ref in order:
        ri = order.index(ref)
        for j, m in enumerate(order):
            if m == ref or m not in piv.columns or ref not in piv.columns:
                continue
            sub = piv[[m, ref]].dropna()
            if len(sub) > 1:
                _, p = wilcoxon_safe(sub[ref].values - sub[m].values)
                add_significance_bracket(ax, j, ri, p)
    ax.set(xlabel='', ylabel=label)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
    if clip:
        ax.set_ylim(*clip)


# encoding: joint vs matched vs optimized (FDE keeps the default y-range)
enc_cmp_sess = (enc_cmp.groupby(['model', 'bat', 'session'])
                .agg(corr=('corr', 'mean'), fde=('fde', 'median')).reset_index())
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
_compare_box(axes[0], enc_cmp_sess, 'corr', 'encoding corr', ENC_ORDER)
_compare_box(axes[1], enc_cmp_sess, 'fde', 'encoding FDE (per-session median)', ENC_ORDER)
fig.suptitle('Encoding: joint vs matched vs optimized Bat_Transformer')
plt.tight_layout(); plt.show()

# decoding: joint vs matched vs optimized
specs = [('err_2d', dec_cmp_ts, '2-D err (m)'), ('err_3d', dec_cmp_ts, '3-D err (m)'),
         ('frechet_2d', fre_cmp, '2-D Fréchet (m)'), ('frechet_3d', fre_cmp, '3-D Fréchet (m)')]
fig, axes = plt.subplots(1, 4, figsize=(19, 4.8))
for ax, (col, src, label) in zip(axes, specs):
    med = src.groupby(['model', 'bat', 'session'])[col].median().reset_index()
    _compare_box(ax, med, col, label, DEC_ORDER)
fig.suptitle('Decoding: joint vs matched vs optimized Bat_Transformer_Decoder')
plt.tight_layout(); plt.show()

## 11 · Notes & next steps

* **Model selection** uses the joint criterion (§5): the Pareto front over
  `(−decode-error, +FDE)`, scored by reference-scaled
  `0.5·(FDE/FDE_REF) − 0.5·(err/ERR_REF)` with `ERR_REF=0.8 m`, `FDE_REF=0.035` — so
  FDE (raw scale ~20× smaller) is up-weighted ~20×. Tune `ERR_REF`/`FDE_REF` (or
  `fde_weight`) to shift the encoding↔decoding trade-off.
* **Hyperparameters** are an Optuna search (§6.1) over `num_layers` (depth, in
  `{1, 2, 4}`, started at 4), `lr`, `weight_decay`, `dropout`, `latent_dim` and
  `ff_dim`, seeded with the prior best config; only `LAM=1.0` is fixed. The input
  read-in takes **raw spike counts** (`INPUT_TRANSFORM='identity'`, no √). The study
  and the selected model are cached under `./out/joint/`.
* **§9** reads the pre-read-out latent on test data and writes interactive 3-D HTML
  for the encode (`h_n`) and decode (`h_b`) geometries to `./out/joint/`.
* **§10** compares the joint model to the dedicated single-task transformers at two
  references: **matched-capacity** baselines trained + cached here, and the
  **optimized** checkpoints from the other notebooks at their native architectures
  (no capacity matching). To also benchmark against the Naive-Bayes / GLM-MAP
  baselines, fold `model='joint'` rows into `decode-position-comparison.ipynb`'s
  `MODEL_ORDER` pipeline (same `err_2d`/`err_3d`/`frechet_*` columns).
* This is **regime 1** (held-out flights within seen sessions); cross-session /
  cross-animal generalisation is out of scope here.